## Despike FDM
Despike using Astro-SCRAPPY's cosmic ray detector.
Reads in fixed-pattern-removed images (and possibly `iris_prep` background subtracted), outputs 2 despiked pickle files (one for each Si IV line).
Next step is background subtraction using `science_background_subtract.ipynb`.

#### Import statements

In [ ]:
%reload_ext autoreload
%autoreload 2
# %matplotlib notebook
%matplotlib inline
import pathlib as pl
import numpy as np
import pickle
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
from matplotlib import colors
import astropy.units as u
from astropy import constants as const
from iris_mosaics import wcs_to_bins, spectral_plot, read_sg_image, read_sg_image_lvl1
import iris_mosaics as iris_fdm
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support
import astroscrappy
quantity_support()

#### File paths of fixed-pattern-removed level 1.1 FDM data

In [ ]:
# Path of level 1.1 fixed-pattern-removed (fpr) and `iris_prep` background-subtracted spectrograph image files
# path = pl.Path(iris_fdm.__file__).parent / 'data' / 'unstitched_mosaic'
path = pl.Path(r'D:\IRIS data\deep_mosaics\20240811')

# path_fdm = path / 'level_11_full_ccd_fixed_pattern_removed'
path_fdm = path / 'level_11_iris_prep_bgsub_fixed_pattern_removed'
files = list(path_fdm.glob('*.fits'))

Compare last two steps (fixed pattern removal and fixed pattern removal plus iris_prep background subtraction)

#### Function to plot data side by side

In [ ]:
def plot_lines_sidebyside(
        si_iv_1394, si_iv_1403, title,
        percentile_min: float = 0,
        percentile_max: float = 100,
        size: tuple = (5,5),
        exp_min = None,
        exp_max = None,
):
    # Set up figure and image grid
    fig = plt.figure(figsize=size)
    grid = ImageGrid(fig,
                     111,          # as in plt.subplot(111)
                     nrows_ncols=(1,2),
                     axes_pad=0.15,
                     # share_all=True,
                     cbar_location="right",
                     cbar_mode="single",
                     cbar_size="20%",
                     cbar_pad=0.15,
                     )

    if exp_min is None:
        exp_min = np.nanpercentile(si_iv_1394, percentile_min)
    if exp_max is None:
        exp_max = np.nanpercentile(si_iv_1394, percentile_max)

    im1 = grid[0].imshow(si_iv_1394, vmin=exp_min, vmax=exp_max)
    im2 = grid[1].imshow(si_iv_1403, vmin=exp_min, vmax=exp_max)
    # grid[0].set_title('1394 $\AA$', color='white')
    # grid[1].set_title('1403 $\AA$', color='white')
    grid[0].set_title('1394 Å')
    grid[1].set_title('1403 Å')
    # fig.suptitle(title, color='white')
    fig.suptitle(title)

    # Colorbar
    grid[~0].cax.colorbar(im2).set_label('DN', rotation=270)
    # grid[~0].cax.toggle_label(True)
    grid[0].invert_yaxis()
    # plt.tight_layout()    # Works, but may still require rect parameter to keep colorbar labels visible
    # plt.show()
    return fig

#### Select Si IV 1394 & 1403 regions
Only work on regions of interest

In [ ]:
# Read in an image in order to see where to crop it down:
file_num = 1111
w_0, hdu_0, _ = read_sg_image(files[file_num],'fuv2')
img_0 = hdu_0[0].data

# Lengths of dimensions of total image
num_y_total = img_0.shape[0]
num_x_total = img_0.shape[1]

# Create masks for the areas of interest (Si IV 1394 & 1403).
# sl_1394 = slice(10,~9), slice(732,~215)
# sl_1403 = slice(10,~9), slice(868,~7)

# For the Aug 2024 mosaic that is twice as wide in the x direction...
x1 = 2 * 732
x2 = 2 * 215
x3 = 2 * 868
x4 = 2 * 8
sl_1394 = slice(10,~9), slice(x1,~x2)
sl_1403 = slice(10,~9), slice(x3,~x4)

# Si IV 1394 and 1403 image dimension lengths
num_y = img_0[sl_1394].shape[0]
num_x_1394 = img_0[sl_1394].shape[1]
num_x_1403 = img_0[sl_1403].shape[1]

# Inspect resulting images (may need to enlarge to see whether the edges are NaNs)
plot_lines_sidebyside(img_0[sl_1394], img_0[sl_1403], '',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10,15),
);

#### Choose fewer files for testing if desired

In [ ]:
# ~~~~~~~~~~~~~~~
# FOR TESTING (apply script to only a few spectrograph images)
# files = files[500:1001]
# ~~~~~~~~~~~~~~~

#### Load selected spectrograph images

In [ ]:
# Length of number of images dimension
num_imgs = len(files)

# Read in spectrograph images, selecting just part of the FUV2 region
sg_1394 = np.empty((num_imgs, num_y, num_x_1394))
sg_1403 = np.empty((num_imgs, num_y, num_x_1403))

for i, file in enumerate(files):
    # Read in full image
    w, hdu, _ = read_sg_image(file,'fuv2')
    sg_img = hdu[0].data
    # Crop to only section of image containing the line of interest
    sg_1394[i] = sg_img[sl_1394]
    sg_1403[i] = sg_img[sl_1403]

#### Function to apply despiking routine
Astroscrappy `detect_cosmics` doesn't remove spikes from the edges very well -- so we added a pad to correct this.

In [ ]:
def despike_cube(cube: np.ndarray, axis: tuple[int, int] = (~1, ~0), **kwargs):
    axis_new = (~1, ~0)
    pad_width = 5
    cube_new = np.moveaxis(cube, source=axis, destination=axis_new)
    # bkg = np.nanmedian(cube_new, axis=0)
    result = np.empty_like(cube)
    result_mask = np.empty_like(cube, dtype=bool)
    shape_orthogonal = tuple(np.array(cube.shape)[:~1])
    for index in np.ndindex(*shape_orthogonal):
        mask_i, result_i = astroscrappy.detect_cosmics(
            indat=np.pad(cube_new[index], pad_width=pad_width),
            # inbkg=np.pad(bkg, pad_width=pad_width),
            **kwargs,
        )
        result[index] = result_i[..., pad_width:~pad_width+1, pad_width:~pad_width+1]
        result_mask[index] = mask_i[..., pad_width:~pad_width+1, pad_width:~pad_width+1]
    result = np.moveaxis(result, source=axis_new, destination=axis)
    result_mask = np.moveaxis(result_mask, source=axis_new, destination=axis)
    return result, result_mask

#### Apply despiker to spectrograph images
The despiker seems to ignore some spikes that are touching NaNs (which were probably infs or something else associated with spikes originally). So set all NaNs to zero just for the despiking, then replace the NaNs at the end... Or rather, replace the NaNs with a very large number so they get taken care of by the despiker? Some images are totally NaNs or half NaNs, so check for those first.

In [ ]:
# Identify NaN elements
nan_mask_1394 = np.isnan(sg_1394)
nan_mask_1403 = np.isnan(sg_1403)

# Percent of each image that is NaNs
percent_nan_1394 = np.sum(nan_mask_1394, axis=(1,2)) / (num_x_1394 * num_y)
percent_nan_1403 = np.sum(nan_mask_1403, axis=(1,2)) / (num_x_1403 * num_y)

# Significant NaNs threshold (percent)
# If the percent of NaNs in an image exceeds this threshold, leave them as NaNs (the image is probably unusable)
# Otherwise, replace them with zero (or a large number?) for despiking
sig_nan = 0.01

# Plot the percent of NaNs in each image, so we can check that the threshold for "significant" NaNs makes sense below
plt.figure()
plt.plot(percent_nan_1394)
plt.plot(percent_nan_1403)
plt.ylim((0,0.01))

In [ ]:
i = 2222
plot_lines_sidebyside(sg_1394[i], sg_1403[i], '',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10,15),
);

If the above threshold looks good, continue.

In [ ]:
# Check if a significant portion of the image is NaNs
all_nan_mask_1394 = percent_nan_1394 > 0.01
all_nan_mask_1403 = percent_nan_1394 > 0.01

# Replace NaNs with zero (or a large number?) for despiking purposes
# sg_1394[nan_mask_1394] = 0.0
# sg_1403[nan_mask_1403] = 0.0
sg_1394[nan_mask_1394] = 16384
sg_1403[nan_mask_1403] = 16384

# Images that exceeded the significant NaNs threshold should stay as NaNs (despiker would do nothing here anyway)
sg_1394[all_nan_mask_1394] = np.nan
sg_1403[all_nan_mask_1403] = np.nan

Despike the data

In [ ]:
%%time
# Apply despiker
sg_1394_dspk, spike_mask_1394 = despike_cube(sg_1394, sigclip=5)
sg_1403_dspk, spike_mask_1403 = despike_cube(sg_1403, sigclip=5)

#### Compare despiked to original data
Check that spectral lines and EEs are not being removed

In [ ]:
# Mean spectrograph image

# Original
sg_1394_mean_img = np.nanmean(sg_1394, axis=0)
sg_1403_mean_img = np.nanmean(sg_1403, axis=0)

# Despiked
sg_1394_dspk_mean_img = np.nanmean(sg_1394_dspk, axis=0)
sg_1403_dspk_mean_img = np.nanmean(sg_1403_dspk, axis=0)

# Mean spectrum

# Original
sg_1394_mean_spectrum = np.nanmean(sg_1394_mean_img, axis=0)
sg_1403_mean_spectrum = np.nanmean(sg_1403_mean_img, axis=0)

# Despiked
sg_1394_dspk_mean_spectrum = np.nanmean(sg_1394_dspk_mean_img, axis=0)
sg_1403_dspk_mean_spectrum = np.nanmean(sg_1403_dspk_mean_img, axis=0)

Plots

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(sg_1394_mean_spectrum, label='original')
plt.plot(sg_1394_dspk_mean_spectrum, label='despiked')
plt.legend()
plt.title('Si IV 1394 $\AA$ mean full disk')

plt.figure(figsize=(10, 3))
plt.plot(sg_1394_mean_spectrum - sg_1394_dspk_mean_spectrum)
plt.title('Si IV 1394 $\AA$ difference')

plt.figure(figsize=(10, 5))
plt.plot(sg_1403_mean_spectrum, label='original')
plt.plot(sg_1403_dspk_mean_spectrum, label='despiked')
plt.legend()
plt.title('Si IV 1403 $\AA$ mean full disk')

plt.figure(figsize=(10, 3))
plt.plot(sg_1403_mean_spectrum - sg_1403_dspk_mean_spectrum)
plt.title('Si IV 1403 $\AA$ difference')

plot_lines_sidebyside(sg_1394_mean_img, sg_1403_mean_img, 'original',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(8, 15),
                      )
plot_lines_sidebyside(sg_1394_dspk_mean_img, sg_1403_dspk_mean_img, 'despiked',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10, 15),
                      )
plot_lines_sidebyside(sg_1394_mean_img - sg_1394_dspk_mean_img, sg_1403_mean_img - sg_1403_dspk_mean_img, 'difference',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10, 15),
                      );

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(sg_1394_mean_spectrum, label='original')
plt.plot(sg_1394_dspk_mean_spectrum, label='despiked')
plt.legend()
plt.title('Si IV 1394 $\AA$ mean full disk')

plt.figure(figsize=(10, 3))
plt.plot(sg_1394_mean_spectrum - sg_1394_dspk_mean_spectrum)
plt.title('Si IV 1394 $\AA$ difference')

plt.figure(figsize=(10, 5))
plt.plot(sg_1403_mean_spectrum, label='original')
plt.plot(sg_1403_dspk_mean_spectrum, label='despiked')
plt.legend()
plt.title('Si IV 1403 $\AA$ mean full disk')

plt.figure(figsize=(10, 3))
plt.plot(sg_1403_mean_spectrum - sg_1403_dspk_mean_spectrum)
plt.title('Si IV 1403 $\AA$ difference')

plot_lines_sidebyside(sg_1394_mean_img, sg_1403_mean_img, 'original',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(8, 15),
                      )
plot_lines_sidebyside(sg_1394_dspk_mean_img, sg_1403_dspk_mean_img, 'despiked',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10, 15),
                      )
plot_lines_sidebyside(sg_1394_mean_img - sg_1394_dspk_mean_img, sg_1403_mean_img - sg_1403_dspk_mean_img, 'difference',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10, 15),
                      );

In [ ]:
# plot_lines_sidebyside(
#     sg_1394_mean_img,
#     sg_1403_mean_img,
#     '',
#     exp_max=45,
#     exp_min=0,
#     size=(7,10),
# );
# .savefig('fixed_pattern_removed_mean_may_2025.png',dpi=300, transparent=True)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(sg_1394_mean_spectrum, label='original')
plt.plot(sg_1394_dspk_mean_spectrum, label='despiked')
plt.legend()
plt.title('Si IV 1394 $\AA$ mean full disk')

plt.figure(figsize=(10,3))
plt.plot(sg_1394_mean_spectrum - sg_1394_dspk_mean_spectrum)
plt.title('Si IV 1394 $\AA$ difference')

plt.figure(figsize=(10,5))
plt.plot(sg_1403_mean_spectrum, label='original')
plt.plot(sg_1403_dspk_mean_spectrum, label='despiked')
plt.legend()
plt.title('Si IV 1403 $\AA$ mean full disk')

plt.figure(figsize=(10,3))
plt.plot(sg_1403_mean_spectrum - sg_1403_dspk_mean_spectrum)
plt.title('Si IV 1403 $\AA$ difference')

plot_lines_sidebyside(sg_1394_mean_img, sg_1403_mean_img, 'original',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(8,15),
)
plot_lines_sidebyside(sg_1394_dspk_mean_img, sg_1403_dspk_mean_img, 'despiked',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10,15),
)
plot_lines_sidebyside(sg_1394_mean_img - sg_1394_dspk_mean_img, sg_1403_mean_img - sg_1403_dspk_mean_img, 'difference',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10,15),
);

#### Inspect example spectrograph image

In [ ]:
# 2024-08-11

i = 7070

# y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
y_1394 = 382
y_1403 = 385

# i = 7070

# y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 364
# y_1403 = 367
#
# i = 8888

# y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 517
# y_1403 = 519

# ~~~~~~~~~~~~~~~~~~~~

# 2014-03-24

# image number
# i = 7000

# y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 438
# y_1403 = 441

# ~~~~~~~~~~~~~~~~~~~~

# # 2019-09-12
#
# # image number
# i = 1000
#
# # y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 82
# y_1403 = 84

# ~~~~~~~~~~~~~~~~~~~~

# # 2019-05-05
#
# # image number
# i = 1000
#
# # y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 517
# y_1403 = 520

In [ ]:
# Half of vertical extent to inspect
dy = 10

plt.figure(figsize=(10, 3))
plt.imshow(sg_1394[i, y_1394 - dy:y_1394 + dy + 1, :],
           vmin=np.nanpercentile(sg_1394[i], 0),
           vmax=np.nanpercentile(sg_1394[i], 99.9),
           origin='lower',)
plt.axhline(y=dy, color='orange')
plt.title('Example EE 1394 $\AA$')

plt.figure(figsize=(10 * (160 / 88), 3))
plt.imshow(sg_1403[i, y_1403 - dy:y_1403 + dy + 1, :],
           vmin=np.nanpercentile(sg_1403[i], 0),
           vmax=np.nanpercentile(sg_1403[i], 99.9),
           origin='lower',)
plt.axhline(y=dy, color='orange')
plt.title('Example EE 1403 $\AA$')

plot_lines_sidebyside(sg_1394[i], sg_1403[i], 'original',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10, 15),
                      )

plot_lines_sidebyside(sg_1394_dspk[i], sg_1403_dspk[i], 'despiked',
                      exp_min=np.nanpercentile(sg_1394[i], 0),
                      exp_max=np.nanpercentile(sg_1394[i], 99.95),
                      size=(10, 15),
                      )

plot_lines_sidebyside(-sg_1394_dspk[i] + sg_1394[i], -sg_1403_dspk[i] + sg_1403[i], 'original - despiked',
                      exp_min=-1,
                      exp_max=(-sg_1394_dspk[i] + sg_1394[i]).max() * .01,
                      size=(10, 15),
                      )

plt.figure(figsize=(8, 4))
plt.plot(np.nanmean(sg_1394[i], axis=0), label='original')
plt.plot(np.nanmean(sg_1394_dspk[i], axis=0), label='despiked')
plt.legend()
plt.title('summed 1394 $\AA$')

plt.figure(figsize=(8 * (160 / 88), 4))
plt.plot(np.nanmean(sg_1403[i], axis=0), label='original')
plt.plot(np.nanmean(sg_1403_dspk[i], axis=0), label='despiked')
plt.legend()
plt.title('summed 1403 $\AA$')

plt.figure(figsize=(8, 4))
for spectrum in range(y_1394 - dy, y_1394 + dy + 1):
    plt.plot(sg_1394[i, spectrum, :], color='tab:blue', label='original')
    plt.plot(sg_1394_dspk[i, spectrum, :], color='tab:orange', label='despiked')
    plt.ylim((None, 4*np.nanmax(sg_1394_dspk[i, spectrum, :])))

plt.figure(figsize=(8 * (160 / 88), 4))
for spectrum in range(y_1403 - dy, y_1403 + dy + 1):
    plt.plot(sg_1403[i, spectrum, :], color='tab:blue', label='original')
    plt.plot(sg_1403_dspk[i, spectrum, :], color='tab:orange', label='despiked')
    # plt.ylim((None, 1.2*np.nanmax(sg_1403_dspk[i, spectrum, :])))

In [ ]:
# 2014-03-24

# image number
i = 10000

# y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
y_1394 = 83
y_1403 = 86

# ~~~~~~~~~~~~~~~~~~~~

# # 2019-09-12
#
# # image number
# i = 996
#
# # y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 101
# y_1403 = 104

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# # 2019-05-05
#
# # image number
# i = 4000
#
# # y-coordinate of example EE in 1394 and 1403 (still tilted so not quite the same number)
# y_1394 = 114
# y_1403 = 117

In [ ]:

# Half of vertical extent to inspect
dy = 5

plt.figure(figsize=(10,3))
plt.imshow(sg_1394[i, y_1394-dy:y_1394+dy+1, :], vmin=np.nanpercentile(sg_1394[i],0), vmax=np.nanpercentile(sg_1394[i],99.9))
plt.axhline(y=dy, color='orange')
plt.title('Example EE 1394 $\AA$')

plt.figure(figsize=(10 * (160/88),3))
plt.imshow(sg_1403[i, y_1403-dy:y_1403+dy+1, :], vmin=np.nanpercentile(sg_1403[i],0), vmax=np.nanpercentile(sg_1403[i],99.9))
plt.axhline(y=dy, color='orange')
plt.title('Example EE 1403 $\AA$')

plot_lines_sidebyside(sg_1394[i], sg_1403[i], 'Si IV original',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10,15),
)

plot_lines_sidebyside(sg_1394_dspk[i], sg_1403_dspk[i], 'Si IV despiked',
                      percentile_min=0,
                      percentile_max=99.95,
                      size=(10,15),
)

plot_lines_sidebyside(-sg_1394_dspk[i] + sg_1394[i], -sg_1403_dspk[i] + sg_1403[i], 'Si IV original - despiked',
                      exp_min=-1,
                      exp_max=(-sg_1403_dspk[i] + sg_1403[i]).max()*.01,
                      size=(10,15),
)

plt.figure(figsize=(8,4))
plt.plot(np.nanmean(sg_1394[i], axis=0), label='original')
plt.plot(np.nanmean(sg_1394_dspk[i], axis=0), label='despiked')
plt.legend()
plt.title('Spectrograph summed 1394 $\AA$')

plt.figure(figsize=(8 * (160/88), 4))
plt.plot(np.nanmean(sg_1403[i], axis=0), label='original')
plt.plot(np.nanmean(sg_1403_dspk[i], axis=0), label='despiked')
plt.legend()
plt.title('Spectrograph summed 1403 $\AA$')

plt.figure(figsize=(8,4))
for spectrum in range(y_1394-dy, y_1394+dy+1):
    plt.plot(sg_1394[i,spectrum,:], color='tab:blue', label='original')
    plt.plot(sg_1394_dspk[i,spectrum,:], color='tab:orange', label='despiked')


plt.figure(figsize=(8 * (160/88), 4))
for spectrum in range(y_1403-dy, y_1403+dy+1):
    plt.plot(sg_1403[i,spectrum,:], color='tab:blue', label='original')
    plt.plot(sg_1403_dspk[i,spectrum,:], color='tab:orange', label='despiked')

#### Images with excessive spikes
Some images have far more spikes, likely due to SAAs, and not all were removed, so we will try to identify the images with excessive spikes and run them through the despiker again but with a lower threshold for spikes to remove the remaining ones.

#### Total spikes for each spectrograph image

In [ ]:
# 2024-08-11
total_spikes_1394 = np.nansum(spike_mask_1394, axis=(1,2))
total_spikes_1403 = np.nansum(spike_mask_1403, axis=(1,2))

plt.figure(figsize=(15,7))
plt.plot(total_spikes_1403)
plt.plot(total_spikes_1394)
plt.axhline(y=14000, color='red', linestyle='--')
plt.axhline(y=1000, color='violet', linestyle='--')
# plt.axhline(y=450, color='tab:blue', linestyle='--')
# plt.axhline(y=250, color='tab:orange', linestyle='--')
plt.xlabel('image number')
plt.ylabel('number of identified spikes')
plt.ylim((10, None))
plt.semilogy()

In [ ]:
# 2014-03-24
total_spikes_1394 = np.nansum(spike_mask_1394, axis=(1,2))
total_spikes_1403 = np.nansum(spike_mask_1403, axis=(1,2))

plt.figure(figsize=(15,7))
plt.plot(total_spikes_1403)
plt.plot(total_spikes_1394)
plt.axhline(y=14000, color='red', linestyle='--')
plt.axhline(y=1000, color='violet', linestyle='--')
# plt.axhline(y=450, color='tab:blue', linestyle='--')
# plt.axhline(y=250, color='tab:orange', linestyle='--')
plt.xlabel('image number')
plt.ylabel('number of identified spikes')
plt.ylim((10, None))
plt.semilogy()

In [ ]:
# 2019-09-12
total_spikes_1394 = np.nansum(spike_mask_1394, axis=(1,2))
total_spikes_1403 = np.nansum(spike_mask_1403, axis=(1,2))

plt.figure(figsize=(15,7))
plt.plot(total_spikes_1403)
plt.plot(total_spikes_1394)
plt.axhline(y=14000, color='red', linestyle='--')
plt.axhline(y=1000, color='violet', linestyle='--')
# plt.axhline(y=450, color='tab:blue', linestyle='--')
# plt.axhline(y=250, color='tab:orange', linestyle='--')
plt.xlabel('image number')
plt.ylabel('number of identified spikes')
plt.ylim((10, None))
plt.semilogy()

#### Remove the spikiest images
The very spikiest images are very difficult to despike. Starting at about 14000 spikes (in 1403 Ang), they begin to develop "islands" of spikes that are not removed by the despiker, even with a lower threshold. These images should be set to NaNs so they don't influence the background subtraction or EE identification later on.
Use the Si IV 1403 $\AA$ to set the excessive spike threshold since it has a larger CCD area/range.
Lowering the threshold to 1000 spikes/image.

In [ ]:
super_excess_spikes_mask = np.zeros(total_spikes_1403.shape, dtype=bool)
super_excess_spikes_mask[total_spikes_1403 >= 1000] = True

plt.figure(figsize=(15,5))
plt.plot(super_excess_spikes_mask)

print(f'Number of images with super excessive spikes: {np.sum(super_excess_spikes_mask)}')

#### Example excessive spike images

In [ ]:
i = 333

plot_lines_sidebyside(sg_1394[super_excess_spikes_mask][i], sg_1403[super_excess_spikes_mask][i], 'original',
                      exp_min=3,
                      exp_max=800,
                      size=(8,15),
)

plot_lines_sidebyside(sg_1394_dspk[super_excess_spikes_mask][i], sg_1403_dspk[super_excess_spikes_mask][i], 'despiked',
                      # exp_min=3,
                      exp_max=100,
                      size=(8,15),
)

plot_lines_sidebyside(spike_mask_1394[super_excess_spikes_mask][i], spike_mask_1403[super_excess_spikes_mask][i], 'spike masks',
                      exp_min=0,
                      exp_max=1,
                      size=(8,15),
)

print(f'Si IV 1394: {np.nansum(spike_mask_1394[super_excess_spikes_mask][i])}')
print(f'Si IV 1403: {np.nansum(spike_mask_1403[super_excess_spikes_mask][i])}')

Set super excessive spiky images to NaNs, and the corresponding spike masks to False.

In [ ]:
sg_1394_dspk[super_excess_spikes_mask] = np.nan
sg_1403_dspk[super_excess_spikes_mask] = np.nan
spike_mask_1394[super_excess_spikes_mask] = False
spike_mask_1403[super_excess_spikes_mask] = False

#### Replace NaNs
Only do this if the NaNs were replaced with zeros earlier. If they were replaced with a large number, then the despiker took care of them.

In [ ]:
# sg_1394_dspk[nan_mask_1394] = np.nan
# sg_1403_dspk[nan_mask_1403] = np.nan

#### Save

In [ ]:
with open(path / 'level_11_fpr_despiked_1394.pickle', 'wb') as fh:
    pickle.dump(sg_1394_dspk, fh)

with open(path / 'level_11_fpr_despiked_1403.pickle', 'wb') as fh:
    pickle.dump(sg_1403_dspk, fh)